# Day 18 — Player List and Player Detail APIs

Build two read-only Django REST Framework endpoints backed by MySQL staging tables:
- `GET /api/players/?search=<name>&page=1&page_size=25`
- `GET /api/players/<player_name>/`

The notebook checks the real database schema, generates API views and app routes, prepares a project URL patch, runs Django's system check, and writes `docs/day18_player_apis.md` plus `reports/day18/` outputs.

Run in the `ipl-player-performance` environment from the IPL project root. It backs up existing `views.py` and `urls.py` before replacing them. **Review the schema inspection output before accepting the generated column mapping.**

**Jupyter note:** This notebook sets `DJANGO_ALLOW_ASYNC_UNSAFE=true` only in the notebook process because Jupyter may run an event loop. Do not add that setting to production Django configuration.

In [1]:
from pathlib import Path
import os, sys, subprocess, importlib.util
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "django_api" / "manage.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "django_api" / "manage.py").exists():
    raise FileNotFoundError("Run from the IPL project root or a subfolder.")
DJANGO_ROOT = PROJECT_ROOT / "django_api"
API_DIR = DJANGO_ROOT / "ipl_api"
DOCS_DIR = PROJECT_ROOT / "docs"
REPORT_DIR = PROJECT_ROOT / "reports" / "day18"
DOCS_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Django project:", DJANGO_ROOT)
print("ipl_api exists:", API_DIR.exists())


Project root: D:\Git\Personal\ipl-player-performance
Django project: D:\Git\Personal\ipl-player-performance\django_api
ipl_api exists: True


In [2]:
modules = {"django": "Django", "rest_framework": "Django REST Framework", "MySQLdb": "mysqlclient", "dotenv": "python-dotenv"}
package_df = pd.DataFrame([{"package": label, "available": importlib.util.find_spec(module) is not None} for module, label in modules.items()])
display(package_df)
if not (DJANGO_ROOT / "manage.py").exists() or not (DJANGO_ROOT / "config" / "settings.py").exists():
    raise FileNotFoundError("Django project files are missing; finish Day 17 first.")
if not (API_DIR / "apps.py").exists():
    raise FileNotFoundError("django_api/ipl_api/apps.py is missing. Create/fix the app before Day 18.")
if not importlib.util.find_spec("rest_framework"):
    raise ModuleNotFoundError("Django REST Framework is not installed.")


,package,available
0,Django,True
1,Django REST Framework,True
2,mysqlclient,True
3,python-dotenv,True


In [3]:
from dotenv import load_dotenv
env_file = PROJECT_ROOT / ".env"
if not env_file.exists():
    raise FileNotFoundError(f"Missing .env file: {env_file}")
load_dotenv(env_file, override=False)
# Jupyter/IPython may have an active event loop. This synchronous ORM/database access is
# limited to this local notebook session; do not set this in production Django settings.
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
sys.path.insert(0, str(DJANGO_ROOT))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
import django
django.setup()
from django.conf import settings
from django.db import connection
DB_ENGINE = settings.DATABASES["default"]["ENGINE"]
DB_NAME = settings.DATABASES["default"].get("NAME")
print("Engine:", DB_ENGINE, "| Database:", DB_NAME)
if DB_ENGINE != "django.db.backends.mysql":
    raise RuntimeError("Django is not configured for MySQL. Fix Day 17 first.")
with connection.cursor() as cursor:
    cursor.execute("SELECT VERSION()")
    MYSQL_VERSION = cursor.fetchone()[0]
print("MySQL version:", MYSQL_VERSION, "| connection: PASS")


Engine: django.db.backends.mysql | Database: ipl_analytics
MySQL version: 8.0.46 | connection: PASS


In [4]:
TABLES = {"batting": "stg_ipl_batting_player_match", "bowling": "stg_ipl_bowling_player_match"}
def get_columns(table):
    with connection.cursor() as cursor:
        cursor.execute("SELECT COLUMN_NAME, DATA_TYPE FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA = DATABASE() AND TABLE_NAME = %s ORDER BY ORDINAL_POSITION", [table])
        return [{"column": r[0], "data_type": r[1]} for r in cursor.fetchall()]
schemas = {kind: get_columns(table) for kind, table in TABLES.items()}
schema_df = pd.DataFrame([{"dataset": kind, "table": TABLES[kind], **col} for kind, cols in schemas.items() for col in cols])
display(schema_df)
if any(not cols for cols in schemas.values()):
    raise RuntimeError("One or both staging tables are missing or empty. Verify Day 16.")
def choose(columns, candidates, label, required=True):
    lookup = {x["column"].lower(): x["column"] for x in columns}
    for candidate in candidates:
        if candidate.lower() in lookup: return lookup[candidate.lower()]
    if required: raise ValueError(f"Cannot infer {label}; review schema output and update candidates.")
    return None
BAT_NAME = choose(schemas["batting"], ["player_name", "batter", "batsman", "striker", "player", "name"], "batting player name")
BOWL_NAME = choose(schemas["bowling"], ["player_name", "bowler", "player", "name"], "bowling player name")
BAT_RUNS = choose(schemas["batting"], ["runs", "batter_runs", "total_runs", "batsman_runs"], "batting runs", False)
BAT_MATCH = choose(schemas["batting"], ["match_id", "match", "match_key", "id"], "batting match identifier", False)
BAT_BALLS = choose(schemas["batting"], ["balls_faced", "balls", "ball_faced"], "balls faced", False)
BAT_FOURS = choose(schemas["batting"], ["fours", "4s"], "fours", False)
BAT_SIXES = choose(schemas["batting"], ["sixes", "6s"], "sixes", False)
BOWL_WICKETS = choose(schemas["bowling"], ["wickets", "wickets_taken", "bowler_wickets"], "bowling wickets", False)
BOWL_RUNS = choose(schemas["bowling"], ["runs_conceded", "runs", "conceded"], "runs conceded", False)
BOWL_OVERS = choose(schemas["bowling"], ["overs", "overs_bowled"], "overs", False)
BOWL_MATCH = choose(schemas["bowling"], ["match_id", "match", "match_key", "id"], "bowling match identifier", False)
detected = {"batting name": BAT_NAME, "bowling name": BOWL_NAME, "runs": BAT_RUNS, "match ID": BAT_MATCH, "balls faced": BAT_BALLS, "fours": BAT_FOURS, "sixes": BAT_SIXES, "wickets": BOWL_WICKETS, "runs conceded": BOWL_RUNS, "overs": BOWL_OVERS, "bowling match ID": BOWL_MATCH}
display(pd.DataFrame([{"setting": k, "detected column": v or "(not present; omitted)"} for k, v in detected.items()]))


,dataset,table,column,data_type
0,batting,stg_ipl_batting_player_match,match_id,bigint
1,batting,stg_ipl_batting_player_match,date,text
2,batting,stg_ipl_batting_player_match,season,text
3,batting,stg_ipl_batting_player_match,venue,text
4,batting,stg_ipl_batting_player_match,city,text
5,batting,stg_ipl_batting_player_match,team_1,text
6,batting,stg_ipl_batting_player_match,team_2,text
7,batting,stg_ipl_batting_player_match,match_type,double
8,batting,stg_ipl_batting_player_match,team,text
9,batting,stg_ipl_batting_player_match,opposition,text


,setting,detected column
0,batting name,player_name
1,bowling name,player_name
2,runs,runs
3,match ID,match_id
4,balls faced,balls_faced
5,fours,fours
6,sixes,sixes
7,wickets,wickets
8,runs conceded,runs_conceded
9,overs,overs


In [5]:
def quote_col(column, kind):
    if column not in {item["column"] for item in schemas[kind]}:
        raise ValueError("Column was not discovered from database metadata.")
    return "`" + column.replace("`", "``") + "`"
source_rows = []
for kind, table in TABLES.items():
    col = BAT_NAME if kind == "batting" else BOWL_NAME
    with connection.cursor() as cursor:
        cursor.execute(f"SELECT COUNT(*) FROM `{table}`")
        rows = cursor.fetchone()[0]
        cursor.execute(f"SELECT COUNT(DISTINCT {quote_col(col, kind)}) FROM `{table}` WHERE {quote_col(col, kind)} IS NOT NULL AND TRIM(CAST({quote_col(col, kind)} AS CHAR)) <> ''")
        players = cursor.fetchone()[0]
    source_rows.append({"dataset": kind, "table": table, "rows": rows, "distinct_nonblank_players": players})
source_counts_df = pd.DataFrame(source_rows)
display(source_counts_df)


,dataset,table,rows,distinct_nonblank_players
0,batting,stg_ipl_batting_player_match,18768,738
1,bowling,stg_ipl_bowling_player_match,14700,577


In [6]:
# Generate views.py from the real schema detected above.
from textwrap import dedent

config = {
    "BATTING_TABLE": TABLES["batting"], "BOWLING_TABLE": TABLES["bowling"],
    "BAT_NAME": BAT_NAME, "BOWL_NAME": BOWL_NAME, "BAT_RUNS": BAT_RUNS,
    "BAT_MATCH": BAT_MATCH, "BAT_BALLS": BAT_BALLS, "BAT_FOURS": BAT_FOURS,
    "BAT_SIXES": BAT_SIXES, "BOWL_WICKETS": BOWL_WICKETS, "BOWL_RUNS": BOWL_RUNS,
    "BOWL_OVERS": BOWL_OVERS, "BOWL_MATCH": BOWL_MATCH,
}
config_lines = "\n".join(f"{key} = {value!r}" for key, value in config.items())

VIEWS_TEMPLATE = dedent('''
from django.db import connection
from django.http import Http404
from rest_framework import status
from rest_framework.response import Response
from rest_framework.views import APIView

# Table and column names detected from the MySQL staging schema.
__SCHEMA_CONFIG__

def q(identifier):
    if not identifier or not identifier.replace("_", "").isalnum():
        raise ValueError("Unsafe database identifier")
    return "`" + identifier.replace("`", "``") + "`"

def fetch_all(sql, params=None):
    with connection.cursor() as cursor:
        cursor.execute(sql, params or [])
        columns = [col[0] for col in cursor.description] if cursor.description else []
        return [dict(zip(columns, row)) for row in cursor.fetchall()]

def number(value):
    if value is None:
        return 0
    try:
        return float(value) if "." in str(value) else int(value)
    except (TypeError, ValueError):
        return value

class PlayerListAPIView(APIView):
    def get(self, request):
        search = (request.query_params.get("search") or "").strip()
        try:
            page = max(1, int(request.query_params.get("page", 1)))
            page_size = min(100, max(1, int(request.query_params.get("page_size", 25))))
        except ValueError:
            return Response({"detail": "page and page_size must be integers."},
                            status=status.HTTP_400_BAD_REQUEST)
        union_sql = (
            f"SELECT {q(BAT_NAME)} AS player_name FROM {q(BATTING_TABLE)} "
            f"WHERE {q(BAT_NAME)} IS NOT NULL AND TRIM(CAST({q(BAT_NAME)} AS CHAR)) <> '' "
            f"UNION SELECT {q(BOWL_NAME)} AS player_name FROM {q(BOWLING_TABLE)} "
            f"WHERE {q(BOWL_NAME)} IS NOT NULL AND TRIM(CAST({q(BOWL_NAME)} AS CHAR)) <> ''"
        )
        filter_sql = " WHERE player_name LIKE %s" if search else ""
        params = [f"%{search}%"] if search else []
        count_rows = fetch_all(
            f"SELECT COUNT(*) AS total FROM ({union_sql}) AS all_players{filter_sql}", params
        )
        total = int(count_rows[0]["total"]) if count_rows else 0
        offset = (page - 1) * page_size
        results = fetch_all(
            f"SELECT player_name FROM ({union_sql}) AS all_players{filter_sql} "
            "GROUP BY player_name ORDER BY player_name LIMIT %s OFFSET %s",
            params + [page_size, offset],
        )
        return Response({"count": total, "page": page, "page_size": page_size,
                         "total_pages": (total + page_size - 1) // page_size, "results": results})

class PlayerDetailAPIView(APIView):
    def get(self, request, player_name):
        bat_fields = ["COUNT(*) AS batting_records"]
        if BAT_MATCH: bat_fields.append(f"COUNT(DISTINCT {q(BAT_MATCH)}) AS matches")
        if BAT_RUNS: bat_fields.append(f"COALESCE(SUM({q(BAT_RUNS)}), 0) AS runs")
        if BAT_BALLS: bat_fields.append(f"COALESCE(SUM({q(BAT_BALLS)}), 0) AS balls_faced")
        if BAT_FOURS: bat_fields.append(f"COALESCE(SUM({q(BAT_FOURS)}), 0) AS fours")
        if BAT_SIXES: bat_fields.append(f"COALESCE(SUM({q(BAT_SIXES)}), 0) AS sixes")
        batting_rows = fetch_all(
            f"SELECT {', '.join(bat_fields)} FROM {q(BATTING_TABLE)} WHERE {q(BAT_NAME)} = %s",
            [player_name])
        batting = batting_rows[0] if batting_rows else {"batting_records": 0}
        bowl_fields = ["COUNT(*) AS bowling_records"]
        if BOWL_MATCH: bowl_fields.append(f"COUNT(DISTINCT {q(BOWL_MATCH)}) AS matches")
        if BOWL_WICKETS: bowl_fields.append(f"COALESCE(SUM({q(BOWL_WICKETS)}), 0) AS wickets")
        if BOWL_RUNS: bowl_fields.append(f"COALESCE(SUM({q(BOWL_RUNS)}), 0) AS runs_conceded")
        if BOWL_OVERS: bowl_fields.append(f"COALESCE(SUM({q(BOWL_OVERS)}), 0) AS overs")
        bowling_rows = fetch_all(
            f"SELECT {', '.join(bowl_fields)} FROM {q(BOWLING_TABLE)} WHERE {q(BOWL_NAME)} = %s",
            [player_name])
        bowling = bowling_rows[0] if bowling_rows else {"bowling_records": 0}
        if int(batting.get("batting_records", 0)) == 0 and int(bowling.get("bowling_records", 0)) == 0:
            raise Http404("Player not found.")
        for section in (batting, bowling):
            for key, value in list(section.items()):
                section[key] = number(value)
        return Response({"player_name": player_name, "batting": batting, "bowling": bowling})
''').strip() + "\n"

views_source = VIEWS_TEMPLATE.replace("__SCHEMA_CONFIG__", config_lines)
compile(views_source, str(API_DIR / "views.py"), "exec")
views_path = API_DIR / "views.py"
backup_path = API_DIR / "views.py.day18_backup"
if views_path.exists() and not backup_path.exists():
    backup_path.write_text(views_path.read_text(encoding="utf-8"), encoding="utf-8")
views_path.write_text(views_source, encoding="utf-8")
print("Generated and syntax-validated:", views_path)
print("Detected schema configuration:", config)
print("Backup:", backup_path if backup_path.exists() else "not needed")


Generated and syntax-validated: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\views.py
Detected schema configuration: {'BATTING_TABLE': 'stg_ipl_batting_player_match', 'BOWLING_TABLE': 'stg_ipl_bowling_player_match', 'BAT_NAME': 'player_name', 'BOWL_NAME': 'player_name', 'BAT_RUNS': 'runs', 'BAT_MATCH': 'match_id', 'BAT_BALLS': 'balls_faced', 'BAT_FOURS': 'fours', 'BAT_SIXES': 'sixes', 'BOWL_WICKETS': 'wickets', 'BOWL_RUNS': 'runs_conceded', 'BOWL_OVERS': 'overs', 'BOWL_MATCH': 'match_id'}
Backup: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\views.py.day18_backup


In [7]:
URLS_TEMPLATE = '''from django.urls import path
from .views import PlayerListAPIView, PlayerDetailAPIView

urlpatterns = [
    path("players/", PlayerListAPIView.as_view(), name="player-list"),
    path("players/<str:player_name>/", PlayerDetailAPIView.as_view(), name="player-detail"),
]
'''
app_urls_path = API_DIR / "urls.py"
if app_urls_path.exists() and not (API_DIR / "urls.py.day18_backup").exists():
    (API_DIR / "urls.py.day18_backup").write_text(app_urls_path.read_text(encoding="utf-8"), encoding="utf-8")
app_urls_path.write_text(URLS_TEMPLATE, encoding="utf-8")
patch = DJANGO_ROOT / "config" / "day18_urls_patch.txt"
patch.write_text(
    "Merge into django_api/config/urls.py\n\n"
    "from django.urls import include, path\n\n"
    "Add this item to urlpatterns: path('api/', include('ipl_api.urls')),\n",
    encoding="utf-8"
)
print("Generated:", app_urls_path)
print("Manual URL integration patch:", patch)
print("Current project urls.py:\n", (DJANGO_ROOT / "config" / "urls.py").read_text(encoding="utf-8"))


Generated: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py
Manual URL integration patch: D:\Git\Personal\ipl-player-performance\django_api\config\day18_urls_patch.txt
Current project urls.py:
 from django.contrib import admin
from django.urls import include, path
from django.views.generic.base import RedirectView

urlpatterns = [
    path(
        "",
        RedirectView.as_view(url="/api/dashboard/", permanent=False),
        name="home",
    ),
    path("api/", include("ipl_api.urls")),
    path("admin/", admin.site.urls),
]



In [8]:
settings_text = (DJANGO_ROOT / "config" / "settings.py").read_text(encoding="utf-8")
app_registered = '"ipl_api"' in settings_text or "'ipl_api'" in settings_text
print("ipl_api registered in INSTALLED_APPS:", app_registered)
if not app_registered:
    print("ACTION REQUIRED: add 'ipl_api' to INSTALLED_APPS.")
check = subprocess.run([sys.executable, str(DJANGO_ROOT / "manage.py"), "check"], cwd=str(DJANGO_ROOT), capture_output=True, text=True)
print(check.stdout)
if check.stderr: print(check.stderr)
print("Django check return code:", check.returncode)


ipl_api registered in INSTALLED_APPS: True
System check identified no issues (0 silenced).

Django check return code: 0


In [9]:
status_rows = [
    {"task": "Django project located", "status": "PASS" if (DJANGO_ROOT / "manage.py").exists() else "FAIL"},
    {"task": "MySQL configured", "status": "PASS" if DB_ENGINE == "django.db.backends.mysql" else "FAIL"},
    {"task": "MySQL connection verified", "status": "PASS" if "MYSQL_VERSION" in globals() else "FAIL"},
    {"task": "Staging schemas inspected", "status": "PASS" if not schema_df.empty else "FAIL"},
    {"task": "Player-name columns detected", "status": "PASS" if BAT_NAME and BOWL_NAME else "FAIL"},
    {"task": "Views generated", "status": "PASS" if views_path.exists() else "FAIL"},
    {"task": "App routes generated", "status": "PASS" if app_urls_path.exists() else "FAIL"},
    {"task": "ipl_api registered", "status": "PASS" if app_registered else "ACTION REQUIRED"},
    {"task": "Project URL include merged", "status": "PENDING MANUAL CHECK"},
    {"task": "Django system check", "status": "PASS" if check.returncode == 0 else "FAIL"},
]
status_df = pd.DataFrame(status_rows)
status_df.to_csv(REPORT_DIR / "task_status.csv", index=False)
package_df.to_csv(REPORT_DIR / "package_availability.csv", index=False)
schema_df.to_csv(REPORT_DIR / "staging_schema.csv", index=False)
source_counts_df.to_csv(REPORT_DIR / "source_player_counts.csv", index=False)
doc_lines = [
    "# Day 18 — Player List and Player Detail APIs", "",
    "## Environment", f"- Django engine: `{DB_ENGINE}`", f"- Database: `{DB_NAME}`", f"- MySQL version: `{MYSQL_VERSION}`", "",
    "## Detected staging schema", schema_df.to_string(index=False), "",
    "## Source counts", source_counts_df.to_string(index=False), "",
    "## Task status", status_df.to_string(index=False), "",
    "## Endpoints", "- `GET /api/players/?search=<name>&page=1&page_size=25`", "- `GET /api/players/<player_name>/`", "",
    "## Manual integration", "- Ensure `ipl_api` is in `INSTALLED_APPS`.", "- Merge `django_api/config/day18_urls_patch.txt` into project URLs.", "- No migrations were run."
]
doc_path = DOCS_DIR / "day18_player_apis.md"
doc_path.write_text("\\n".join(doc_lines) + "\\n", encoding="utf-8")
print("Documentation:", doc_path)
print("Reports:", REPORT_DIR)
display(status_df)


Documentation: D:\Git\Personal\ipl-player-performance\docs\day18_player_apis.md
Reports: D:\Git\Personal\ipl-player-performance\reports\day18


,task,status
0,Django project located,PASS
1,MySQL configured,PASS
2,MySQL connection verified,PASS
3,Staging schemas inspected,PASS
4,Player-name columns detected,PASS
5,Views generated,PASS
6,App routes generated,PASS
7,ipl_api registered,PASS
8,Project URL include merged,PENDING MANUAL CHECK
9,Django system check,PASS
